# Urdu Question Generation - training run

From-scratch BiLSTM encoder-decoder with Bahdanau attention. The real code lives in the
repo's `src/` package; this notebook only clones it and runs the modules so the code is
identical locally and on Kaggle.

**Before running - Settings panel (right):** Accelerator = **GPU** (P100 or T4 x2),
Internet = **On**.

Order: clone + deps -> data prep -> tokenizer -> debug gate -> full train -> evaluate -> zip.

In [ ]:
# 1. Get the code and install the few deps the Kaggle image lacks.
!git clone --depth 1 https://github.com/Hanzala-12/urdu-question-generation.git
%cd urdu-question-generation
!pip -q install sentencepiece sacrebleu rouge-score
import torch
print('torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU!')

In [ ]:
# 2. Task 1 + Task 2 - data prep (needs Internet: On) and the 8k SentencePiece tokenizer.
!python -m src.data_prep
!python -m src.spm_train
!wc -l data/*.tsv

In [ ]:
# 3. Debug gate - 10k pairs, 1 epoch. The loss MUST fall or there is a bug to fix first.
!python -m src.train --debug

In [ ]:
# 4. Task 3 - full training. ~1-2 h on a P100 (batch 64, 15 epochs).
!python -m src.train --epochs 15 --batch-size 64

In [ ]:
# 5. Task 4 - evaluation on UQA-valid and Wiki-UQA (greedy + beam).
!python -m src.evaluate --split both --beam-max 3000
import json; print(json.dumps(json.load(open('results/metrics.json')), indent=2, ensure_ascii=False))

In [ ]:
# 6. Bundle artifacts/ + results/ for download back into the local repo.
import shutil
shutil.copytree('artifacts', '/kaggle/working/outputs/artifacts', dirs_exist_ok=True)
shutil.copytree('results', '/kaggle/working/outputs/results', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/outputs')
print('wrote /kaggle/working/outputs.zip')
!ls -lhR /kaggle/working/outputs